# ClinIQ — Exploratory Analysis
**Heart Failure GDMT Gap Detection on Synthetic EHR Data**

This notebook explores the synthetic Synthea patient population, validates the gap detection logic,
and generates key statistics for the Power BI dashboard.

**Data**: Synthetic patients generated by Synthea — no real PHI.
**Stack**: Snowflake → dbt → Python (pandas, matplotlib, seaborn)

In [ ]:
import os
import sys
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from dotenv import load_dotenv
import snowflake.connector

load_dotenv('../.env')
sns.set_theme(style='whitegrid', palette='muted')
print('Setup complete.')

## 1. Connect to Snowflake and load mart tables

In [ ]:
conn = snowflake.connector.connect(
    account=os.getenv('SNOWFLAKE_ACCOUNT'),
    user=os.getenv('SNOWFLAKE_USER'),
    password=os.getenv('SNOWFLAKE_PASSWORD'),
    warehouse=os.getenv('SNOWFLAKE_WAREHOUSE', 'COMPUTE_WH'),
    database=os.getenv('SNOWFLAKE_DATABASE', 'CLINIQ'),
    schema='MARTS',
)

# Load gap registry
gap_df = pd.read_sql('SELECT * FROM MART_GAP_REGISTRY', conn)
summary_df = pd.read_sql('SELECT * FROM MART_QUALITY_SUMMARY', conn)

print(f'Gap registry: {len(gap_df):,} rows')
print(f'Columns: {list(gap_df.columns)}')
gap_df.head()

## 2. Overall quality measure summary

In [ ]:
total = len(gap_df)
gap_count = gap_df['has_care_gap'].sum()
treated_count = (~gap_df['has_care_gap']).sum()
gap_rate = round(gap_count / total * 100, 1)

print('='*50)
print('  ClinIQ — HF GDMT Gap Analysis Summary')
print('='*50)
print(f'  Total HF patients in cohort : {total:,}')
print(f'  Receiving GDMT              : {treated_count:,} ({100-gap_rate}%)')
print(f'  Patients with care gap      : {gap_count:,} ({gap_rate}%)')
print('='*50)

## 3. Gap breakdown by gap reason

In [ ]:
gap_reason_counts = gap_df['gap_reason'].value_counts()

fig, ax = plt.subplots(figsize=(10, 5))
gap_reason_counts.plot(kind='barh', ax=ax, color='steelblue')
ax.set_title('Gap Breakdown by Reason', fontsize=14, fontweight='bold')
ax.set_xlabel('Number of Patients')
ax.set_ylabel('')
for i, v in enumerate(gap_reason_counts.values):
    ax.text(v + 1, i, str(v), va='center', fontsize=10)
plt.tight_layout()
plt.savefig('../dashboard/gap_reason_breakdown.png', dpi=150)
plt.show()

## 4. Gap rate by gender

In [ ]:
gender_gaps = gap_df.groupby('gender')['has_care_gap'].agg(['sum', 'count'])
gender_gaps['gap_rate_pct'] = round(gender_gaps['sum'] / gender_gaps['count'] * 100, 1)
gender_gaps.columns = ['gap_count', 'total', 'gap_rate_pct']

fig, ax = plt.subplots(figsize=(7, 4))
gender_gaps['gap_rate_pct'].plot(kind='bar', ax=ax, color=['#4e79a7', '#f28e2b'], edgecolor='white')
ax.set_title('Care Gap Rate by Gender', fontsize=14, fontweight='bold')
ax.set_ylabel('Gap Rate (%)')
ax.set_xticklabels(gender_gaps.index, rotation=0)
for i, v in enumerate(gender_gaps['gap_rate_pct']):
    ax.text(i, v + 0.3, f'{v}%', ha='center', fontsize=11)
plt.tight_layout()
plt.show()
print(gender_gaps)

## 5. Age distribution — gaps vs. treated

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
gap_df[gap_df['has_care_gap']]['age'].hist(
    bins=20, ax=ax, alpha=0.6, color='tomato', label='Care Gap'
)
gap_df[~gap_df['has_care_gap']]['age'].hist(
    bins=20, ax=ax, alpha=0.6, color='steelblue', label='On GDMT'
)
ax.set_title('Age Distribution: Patients With vs. Without Care Gap', fontsize=13, fontweight='bold')
ax.set_xlabel('Age')
ax.set_ylabel('Patient Count')
ax.legend()
plt.tight_layout()
plt.show()

print('Average age (gap patients):    ', round(gap_df[gap_df['has_care_gap']]['age'].mean(), 1))
print('Average age (treated patients):', round(gap_df[~gap_df['has_care_gap']]['age'].mean(), 1))

## 6. Gap rate by race/ethnicity

In [ ]:
race_gaps = (
    gap_df.groupby('race')['has_care_gap']
    .agg(['sum', 'count'])
    .rename(columns={'sum': 'gap_count', 'count': 'total'})
)
race_gaps['gap_rate_pct'] = round(race_gaps['gap_count'] / race_gaps['total'] * 100, 1)
race_gaps = race_gaps.sort_values('gap_rate_pct', ascending=True)

fig, ax = plt.subplots(figsize=(9, 5))
race_gaps['gap_rate_pct'].plot(kind='barh', ax=ax, color='#59a14f')
ax.set_title('Care Gap Rate by Race', fontsize=14, fontweight='bold')
ax.set_xlabel('Gap Rate (%)')
plt.tight_layout()
plt.show()
print(race_gaps)

## 7. Validate gap logic against raw medication data
Spot-check: a randomly sampled 'gap' patient should have no GDMT in the medications table.

In [ ]:
# Sample one gap patient
gap_patient = gap_df[gap_df['has_care_gap']]['patient_id'].sample(1).values[0]
print(f'Spot-checking gap patient: {gap_patient}')

meds = pd.read_sql(
    f"SELECT patient_id, rxnorm_code, medication_name, is_active, days_since_start "
    f"FROM CLINIQ.STAGING.STG_MEDICATIONS "
    f"WHERE patient_id = '{gap_patient}'",
    conn
)

gdmt_codes = ['866511', '854901', '200031', '29046', '214354', '18867']
gdmt_meds = meds[meds['rxnorm_code'].isin(gdmt_codes)]

print(f'Total medications for this patient: {len(meds)}')
print(f'GDMT medications found: {len(gdmt_meds)}')
print('✓ Gap flag is correct — no active GDMT found.' if len(gdmt_meds) == 0 else '⚠ GDMT found — check gap logic.')
meds.head(10)

## 8. Close connection

In [ ]:
conn.close()
print('Analysis complete. Connection closed.')